In [1]:
import os
import numpy as np 
import pandas as pd
import time 
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt
import random

# Check if GPU is available 
is_gpu = torch.cuda.is_available()
device = torch.device("cuda" if is_gpu else "cpu")
print(is_gpu)

True


## Load modules from AI4PDEs

In [2]:
from AI4PDEs_utils  import create_tensors_3D, create_tensors_2D, get_weights_linear_2D, get_weights_linear_3D
from AI4PDEs_bounds import boundary_condition_3D_u, boundary_condition_3D_v, boundary_condition_3D_w
from AI4PDEs_bounds import boundary_condition_3D_p, boundary_condition_3D_k, boundary_condition_3D_cw

ModuleNotFoundError: No module named 'AI4PDEs_utils'

## Initialise numerical parameter

In [ ]:
dt = 0.5                                # Time step (s)
dx = 1.0 ; dy = 1.0 ; dz = 1.0          # Grid size (m)
nu = 0.25                               # Viscosity coefficient
ub = -1.0                               # Inflow speed (m/s) ***negative means the inflow from left to right***
nx = 256 ; ny = 32 ; nz = 256           # Grid point
nlevel = int(math.log(nz, 2)) + 1 -2      # Multigrid level
ntime = 500                             # Time step
n_out = 100                             # Time step to save results 
n_check = 50                            # Time step to check residual
iteration = 5                           # Multigrid iteration
filepath = 'test_buildings'             # filepath to save results 
T_stat = True                           # Generate time histories at specific points  
L_save = True                           # Save results
bias_initializer = torch.tensor([0.0])  # Initial bias as 0 for NNs 

In [ ]:
if not os.path.exists(filepath):
    os.makedirs(filepath)

## Initialise numerical parameter

In [ ]:
[w1, w2, w3, w4, wA, w_res, diag] = get_weights_linear_3D(dx)

## Establish AI4CFD Neural Network

In [ ]:
class AI4Urban(nn.Module):
    """docstring for AI4Urban"""
    def __init__(self):
        super(AI4Urban, self).__init__()
        # self.arg = arg
        self.xadv = nn.Conv3d(1, 1, kernel_size=3, stride=1, padding=0)
        self.yadv = nn.Conv3d(1, 1, kernel_size=3, stride=1, padding=0)
        self.zadv = nn.Conv3d(1, 1, kernel_size=3, stride=1, padding=0)
        self.diff = nn.Conv3d(1, 1, kernel_size=3, stride=1, padding=0)

        self.A = nn.Conv3d(1, 1, kernel_size=3, stride=1, padding=0)
        self.res = nn.Conv3d(1, 1, kernel_size=2, stride=2, padding=0)  
        self.prol = nn.Sequential(nn.Upsample(scale_factor=2, mode='nearest'),)

        self.A.weight.data = wA
        self.res.weight.data = w_res
        self.diff.weight.data = w1
        self.xadv.weight.data = w2
        self.yadv.weight.data = w3
        self.zadv.weight.data = w4

        self.A.bias.data = bias_initializer
        self.res.bias.data = bias_initializer
        self.diff.bias.data = bias_initializer
        self.xadv.bias.data = bias_initializer
        self.yadv.bias.data = bias_initializer
        self.zadv.bias.data = bias_initializer

    def solid_body(self, values_u, values_v, values_w, sigma, dt):
        values_u = values_u / (1 + dt * sigma) 
        values_v = values_v / (1 + dt * sigma) 
        values_w = values_w / (1 + dt * sigma) 
        return values_u, values_v, values_w

    def F_cycle_MG(self, values_uu, values_vv, values_ww, values_p, values_pp, iteration, diag, dt, nlevel):
        b = -(self.xadv(values_uu) + self.yadv(values_vv) + self.zadv(values_ww)) / dt
        for MG in range(iteration):
              w = torch.zeros((1,1,1,1,1), device=device)
              r = self.A(boundary_condition_3D_p(values_p, values_pp)) - b 
              r_s = []  
              r_s.append(r)
              for i in range(1,nlevel-1):
                     r = self.res(r)
                     r_s.append(r)
              for i in reversed(range(1,nlevel-1)):
                     ww = boundary_condition_3D_cw(w)
                     w = w - self.A(ww) / diag + r_s[i] / diag
                     w = self.prol(w)         
              values_p = values_p - w
              values_p = values_p - self.A(boundary_condition_3D_p(values_p, values_pp)) / diag + b / diag
        return values_p, w, r

    def PG_vector(self, values_uu, values_vv, values_ww, values_u, values_v, values_w, ADx_u, ADy_u, ADz_u, ADx_v, ADy_v, ADz_v, ADx_w, ADy_w, ADz_w, AD2_u, AD2_v, AD2_w):
        # k_u = 0.1 * dx * torch.abs(1/3 * dx**-3 * (torch.abs(values_u) * dx + torch.abs(values_v) * dy + torch.abs(values_w) * dz) * AD2_u) / \
        #     (1e-03 + (torch.abs(ADx_u) * dx**-3 + torch.abs(ADy_u) * dx**-3 + torch.abs(ADz_u) * dx**-3) / 3)

        # k_v = 0.1 * dy * torch.abs(1/3 * dx**-3 * (torch.abs(values_u) * dx + torch.abs(values_v) * dy + torch.abs(values_w) * dz) * AD2_v) / \
        #     (1e-03 + (torch.abs(ADx_v) * dx**-3 + torch.abs(ADy_v) * dx**-3 + torch.abs(ADz_v) * dx**-3) / 3)

        # k_w = 0.1 * dz * torch.abs(1/3 * dx**-3 * (torch.abs(values_u) * dx + torch.abs(values_v) * dy + torch.abs(values_w) * dz) * AD2_w) / \
        #     (1e-03 + (torch.abs(ADx_w) * dx**-3 + torch.abs(ADy_w) * dx**-3 + torch.abs(ADz_w) * dx**-3) / 3)

        # k_u = torch.clamp_max(k_u, 2.0) / (1 + dt * sigma) 
        # k_v = torch.clamp_max(k_v, 2.0) / (1 + dt * sigma) 
        # k_w = torch.clamp_max(k_w, 2.0) / (1 + dt * sigma) 

        k_u = torch.ones_like(values_u) * nu
        k_v = torch.ones_like(values_u) * nu
        k_w = torch.ones_like(values_u) * nu

        k_uu = boundary_condition_3D_k(k_u) 
        k_vv = boundary_condition_3D_k(k_v)  
        k_ww = boundary_condition_3D_k(k_w)

        k_x = 0.5 * (k_u * AD2_u + self.diff(values_uu * k_uu) - values_u * self.diff(k_uu))
        k_y = 0.5 * (k_v * AD2_v + self.diff(values_vv * k_vv) - values_v * self.diff(k_vv))
        k_z = 0.5 * (k_w * AD2_w + self.diff(values_ww * k_ww) - values_w * self.diff(k_ww))
        return k_x, k_y, k_z

    def forward(self, values_u, values_uu, values_v, values_vv, values_w, values_ww, values_p, values_pp, b_uu, b_vv, b_ww, dt, iteration):
    # Padding velocity vectors 
        values_uu = boundary_condition_3D_u(values_u,values_uu,ub)  
        values_vv = boundary_condition_3D_v(values_v,values_vv,ub)   
        values_ww = boundary_condition_3D_w(values_w,values_ww,ub) 
        values_pp = boundary_condition_3D_p(values_p,values_pp) 

        Grapx_p = self.xadv(values_pp) * dt ; Grapy_p = self.yadv(values_pp) * dt ; Grapz_p = self.zadv(values_pp) * dt  
        ADx_u = self.xadv(values_uu) ; ADy_u = self.yadv(values_uu) ; ADz_u = self.zadv(values_uu)
        ADx_v = self.xadv(values_vv) ; ADy_v = self.yadv(values_vv) ; ADz_v = self.zadv(values_vv)
        ADx_w = self.xadv(values_ww) ; ADy_w = self.yadv(values_ww) ; ADz_w = self.zadv(values_ww)
        AD2_u = self.diff(values_uu) ; AD2_v = self.diff(values_vv) ; AD2_w = self.diff(values_ww)
    # First step for solving uvw
        [k_x,k_y,k_z] = self.PG_vector(values_uu, values_vv, values_ww, values_u, values_v, values_w, 
                                       ADx_u, ADy_u, ADz_u, ADx_v, ADy_v, ADz_v, ADx_w, ADy_w, ADz_w, AD2_u, AD2_v, AD2_w)

        b_u = values_u + 0.5 * (nu * k_x * dt - values_u * ADx_u * dt - values_v * ADy_u * dt - values_w * ADz_u * dt) - Grapx_p
        b_v = values_v + 0.5 * (nu * k_y * dt - values_u * ADx_v * dt - values_v * ADy_v * dt - values_w * ADz_v * dt) - Grapy_p
        b_w = values_w + 0.5 * (nu * k_z * dt - values_u * ADx_w * dt - values_v * ADy_w * dt - values_w * ADz_w * dt) - Grapz_p
    # Solid body
        [b_u, b_v, b_w] = self.solid_body(b_u, b_v, b_w, sigma, dt)
    # Padding velocity vectors 
        b_uu = boundary_condition_3D_u(b_u,b_uu,ub) 
        b_vv = boundary_condition_3D_v(b_v,b_vv,ub)  
        b_ww = boundary_condition_3D_w(b_w,b_ww,ub) 

        ADx_u = self.xadv(b_uu) ; ADy_u = self.yadv(b_uu) ; ADz_u = self.zadv(b_uu)
        ADx_v = self.xadv(b_vv) ; ADy_v = self.yadv(b_vv) ; ADz_v = self.zadv(b_vv)
        ADx_w = self.xadv(b_ww) ; ADy_w = self.yadv(b_ww) ; ADz_w = self.zadv(b_ww)
        AD2_u = self.diff(b_uu) ; AD2_v = self.diff(b_vv) ; AD2_w = self.diff(b_ww)

        [k_x,k_y,k_z] = self.PG_vector(b_uu, b_vv, b_ww, b_u, b_v, b_w, 
                                       ADx_u, ADy_u, ADz_u, ADx_v, ADy_v, ADz_v, ADx_w, ADy_w, ADz_w, AD2_u, AD2_v, AD2_w)   
    # Second step for solving uvw   
        values_u = values_u + nu * k_x * dt - b_u * ADx_u * dt - b_v * ADy_u * dt - b_w * ADz_u * dt - Grapx_p  
        values_v = values_v + nu * k_y * dt - b_u * ADx_v * dt - b_v * ADy_v * dt - b_w * ADz_v * dt - Grapy_p  
        values_w = values_w + nu * k_z * dt - b_u * ADx_w * dt - b_v * ADy_w * dt - b_w * ADz_w * dt - Grapz_p
    # Solid body
        [values_u, values_v, values_w] = self.solid_body(values_u, values_v, values_w, sigma, dt)
    # pressure
        values_uu = boundary_condition_3D_u(values_u,values_uu,ub)
        values_vv = boundary_condition_3D_v(values_v,values_vv,ub) 
        values_ww = boundary_condition_3D_w(values_w,values_ww,ub)  
        [values_p, w ,r] = self.F_cycle_MG(values_uu, values_vv, values_ww, values_p, values_pp, iteration, diag, dt, nlevel)
    # Pressure gradient correction    
        values_pp = boundary_condition_3D_p(values_p, values_pp)        
        values_u = values_u - self.xadv(values_pp) * dt ; values_v = values_v - self.yadv(values_pp) * dt ; values_w = values_w - self.zadv(values_pp) * dt      
    # Solid body
        [values_u, values_v, values_w] = self.solid_body(values_u, values_v, values_w, sigma, dt)
        return values_u, values_v, values_w, values_p, w, r

## Send the model to GPU

In [ ]:
AI4Urban = AI4Urban().to(device)

## Create initial tensors 

In [ ]:
values_u, values_v, values_w, values_p, values_uu, values_vv, values_ww, values_pp, b_uu, b_vv, b_ww = create_tensors_3D(nx, ny, nz)

## Loading buildings mesh

In [ ]:
sigma = torch.zeros_like(values_u)
for i in range(nx):
    for j in range(nz):
        if ((i-64)**2+(j-128)**2)**0.5 <=10:
            sigma[0,0,j,:,i] = 1e08

# sigma[0,0,0:10,118:138,118:138] = 1e08
plt.figure(figsize=(12, 6))
plt.subplot(211)
plt.imshow(sigma[0,0,128,:,:].cpu())
plt.colorbar()
plt.subplot(212)
plt.imshow(sigma[0,0,:,16,:].cpu())
plt.colorbar()
plt.gca().invert_yaxis()

## Run AI4Urban solver

In [ ]:
start = time.time()
print("========================================================================")
print("Welcome to AI4CFD solver that will generate flow past buildings for you!")
print("========================================================================")
print("Summarising basic numerical setup before running AI4CFD code............")
print(f'inflow speed from left to right --- {-ub} (m/s)')
print(f'Time step ------------------------- {dt} (s)')
print(f'Grid size ------------------------- {dx} (m)')
if L_save == True:
    print("You are saving spatial results!")
print("========================================================================")
print("Hello World, AI4CFD is running now!")
with torch.no_grad():
    for itime in range(1,ntime+1):
        [values_u,values_v,values_w,values_p,w,r] = AI4Urban(values_u,values_uu,values_v,
                                                             values_vv,values_w,values_ww,
                                                             values_p,values_pp,b_uu,b_vv,b_ww,
                                                             dt, iteration)        
        if itime % n_check == 0:
            print('Time step:', itime, 'Pressure residual:',"{:.5f}".format(np.max(np.abs(w.cpu().detach().numpy()))))  
        if np.max(np.abs(w.cpu().detach().numpy())) > 80000.0:
            print('Not converged !!!!!!')
            break
        if L_save and itime % n_out == 0:
            np.save(filepath+"/u"+str(itime), arr=values_u.cpu().detach().numpy())
            np.save(filepath+"/v"+str(itime), arr=values_v.cpu().detach().numpy()) 
            np.save(filepath+"/w"+str(itime), arr=values_w.cpu().detach().numpy()) 
end = time.time()
print('Elapsed time:', end - start)
print("Goodbye World, AI4CFD is sleeping now!")

## Visualise u component velocity in x direction

In [ ]:
plt.figure(figsize=(12, 6))
plt.subplot(211)
plt.imshow(-values_u[0,0,128,:,:].cpu())
plt.colorbar()
plt.title('u component velocity (m/s)')
plt.subplot(212)
plt.imshow(-values_u[0,0,:,16,:].cpu())
plt.colorbar()
plt.gca().invert_yaxis()
plt.title('u component velocity (m/s)')